# FIFA World Cup Analytics (1930–2026)
### Historical Trends, Match Insights & Predictive Analysis

**Notebook Phase:** 1 of 12 — Introduction
**Blueprint version:** v1.2 (locked — see `FIFA_WC_Project_Master_Blueprint.md`)

---


## 1.1 History of the FIFA World Cup

The FIFA World Cup is football's premier international tournament, held every four years since 1930 (with breaks in 1942 and 1946 due to World War II). What began in Montevideo, Uruguay with 13 teams has grown into a 48-team global event as of 2026 — the tournament analyzed in this notebook is the 23rd edition in history, co-hosted for the first time across three nations (USA, Canada, Mexico).

Across nearly a century, the tournament has reflected football's tactical evolution — from the low-scoring, defensively organized eras of the mid-20th century to the faster, more physically demanding modern game — while also mirroring geopolitical shifts: national teams that no longer exist (Czechoslovakia, West Germany, the Soviet Union, Yugoslavia, Zaire) appear alongside teams still competing today under different names or as successor nations.

This history is exactly why the dataset requires careful cleaning before analysis (Phase 3) — team identities, stage naming conventions, and even encoding of country names have changed or been recorded inconsistently across 96 years of record-keeping.


## 1.2 Objectives

This project aims to build an end-to-end, evidence-based analysis of World Cup history, structured so that every visualization answers a specific question and every question ties to a broader football-intelligence conclusion. Concretely, the notebook will:

1. **Consolidate** five separate data sources (match results, penalty shootouts, 2026 squads, historical country summaries, ISO codes) into a single clean, feature-rich dataset.
2. **Quantify** tournament evolution — scoring trends, competitiveness, format changes — across eight EDA categories and ~58 visualizations.
3. **Test** specific football hypotheses statistically (home advantage, first-goal importance, era-based scoring shifts) rather than relying on visual impression alone.
4. **Model** match outcomes and goal totals using machine learning, while being explicit about what these models can and cannot reliably predict.
5. **Package** the findings into KPIs, a football-intelligence Q&A layer, and (as separate post-notebook deliverables) a Power BI dashboard, written report, and presentation.

This notebook itself stays scoped to analysis — the GitHub repository, README, dashboard, report, and presentation are built afterward, from its finished output, not folded into it.


## 1.3 Dataset Description

Five source files are used. Each is introduced here; full schemas and cleaning decisions are covered in Phases 2–3.

| # | File | Contents | Size | Role |
|---|---|---|---|---|
| 1 | `WorldCupMatches-selected-columns.csv` | Every World Cup match, 1930–2026: year, stage, stadium, city, teams, goals, win conditions | 1,074 matches | Backbone of the entire analysis |
| 2 | `wc_penshoototu_perfected.xlsx` | Kick-by-kick penalty shootout data: shooter, goalkeeper, kick number, score before kick, pressure flags, outcome | 352 kicks, 12 shootout-decided tournaments | Powers the Penalty Intelligence section |
| 3 | `world_cup_2026_squads_new.csv` | 2026 squad rosters: player name, age, country, position, club, jersey number | 1,248 players, 48 countries | Powers the 2026 Squad Analysis section (present-day snapshot only) |
| 4 | `2026_team_summaries.csv` | Per-country lifetime aggregates: average 2026 squad age, clubs represented, historical matches played, historical goals scored | 48 countries | Cross-referenced against squad data for "pedigree vs. current squad" questions |
| 5 | `country_iso_mapping.csv` | Country name → ISO-3 code | 48 countries | Enables choropleth/map visuals |

**Two lookup tables were built manually** to fill genuine gaps in the source data (documented in full in the blueprint's Data Reality Check):
- `host_country_lookup.csv` — Year → Host Nation(s) for all 23 tournaments, including the 2002 and 2026 co-hosts.
- A `Country → Continent` mapping, built during Feature Engineering (Phase 4), to support continent-level comparisons.

**What this data cannot answer**, stated upfront rather than discovered mid-analysis: there is no attendance data, no disciplinary/cards data, no minute-by-minute goal timing, no travel-distance data, and no historical player-level statistics (goals, assists, awards) beyond the 2026 squad snapshot. These limitations are treated as fixed constraints on scope, not gaps to be quietly filled in.


## 1.4 Business (Football Intelligence) Questions — Preview

The full analysis is ultimately organized around twelve questions, each answered later using specific charts and statistical tests built earlier in the notebook (Section 8 of the blueprint). They are previewed here so every subsequent phase can be read with its purpose in mind:

1. Which country dominates the World Cup, and by how much?
2. Has home advantage genuinely helped host nations?
3. Has scoring increased or decreased over 96 years?
4. Which decade produced the most "exciting" (highest-scoring) football?
5. Which continent dominates when adjusted for hosting?
6. How important is scoring first?
7. How many champions have gone back-to-back or repeated titles?
8. Does tournament expansion (more teams) increase or dilute goals per match?
9. Are penalty shootouts a lottery, or does kick order/pressure show a real pattern?
10. Is there a "changing of the guard" — are traditional powerhouses still dominant, or has the sport globalized?
11. Do younger or older 2026 squads correlate with stronger historical pedigree?
12. Which stadiums and cities are the true "theatres" of World Cup history?

---
**Next: Phase 2 — Data Loading.** All five files will be loaded and given an initial structural audit (shape, dtypes, missingness) before any cleaning is applied.


---
# Phase 2: Data Loading

**Goal:** load all five source files plus the manually-built host-country lookup, and run a purely structural audit — shape, dtypes, missing values, duplicates — of each. No values are changed in this phase. Every issue found here is fixed deliberately in Phase 3, not silently.


In [1]:
import pandas as pd
import numpy as np
import openpyxl
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

DATA_DIR = 'data/raw'
LOOKUP_DIR = 'data/lookups'

print("Environment ready.")
print("pandas:", pd.__version__)

Environment ready.
pandas: 3.0.2


## 2.1 Load Matches (`wc_matches.csv`)

The backbone dataset — every World Cup match, 1930–2026.

In [2]:
df_matches = pd.read_csv(f'{DATA_DIR}/wc_matches.csv')
print("Shape:", df_matches.shape)
df_matches.info()

Shape: (1074, 10)
<class 'pandas.DataFrame'>
RangeIndex: 1074 entries, 0 to 1073
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             1074 non-null   float64
 1   Datetime         980 non-null    str    
 2   Stage            1074 non-null   str    
 3   Stadium          1074 non-null   str    
 4   City             1074 non-null   str    
 5   Home Team Name   1074 non-null   str    
 6   Home Team Goals  1074 non-null   float64
 7   Away Team Goals  1074 non-null   float64
 8   Away Team Name   1074 non-null   str    
 9   Win conditions   1074 non-null   str    
dtypes: float64(3), str(7)
memory usage: 84.0 KB


In [3]:
df_matches.head(3)

,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions
0,1930.0,13 Jul 1930 - 15:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,
1,1930.0,13 Jul 1930 - 15:00,Group 4,Parque Central,Montevideo,USA,3.0,0.0,Belgium,
2,1930.0,14 Jul 1930 - 12:45,Group 2,Parque Central,Montevideo,Yugoslavia,2.0,1.0,Brazil,


## 2.2 Load Penalty Shootouts (`wc_penalty_shootouts.xlsx`)

Kick-by-kick shootout detail, loaded from Excel via `openpyxl`.

In [4]:
wb = openpyxl.load_workbook(f'{DATA_DIR}/wc_penalty_shootouts.xlsx', data_only=True)
ws = wb['Sheet1']
rows = list(ws.iter_rows(values_only=True))
df_shootouts = pd.DataFrame(rows[1:], columns=rows[0])

print("Shape:", df_shootouts.shape)
df_shootouts.info()

Shape: (352, 13)
<class 'pandas.DataFrame'>
RangeIndex: 352 entries, 0 to 351
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Tournament_Year    352 non-null    int64
 1   Match_Stage        352 non-null    str  
 2   Team_A             352 non-null    str  
 3   Team_B             352 non-null    str  
 4   Shooter_Team       352 non-null    str  
 5   Shooter_Name       352 non-null    str  
 6   Goalkeeper_Name    352 non-null    str  
 7   Kick_Number        352 non-null    int64
 8   Team_Kick_Number   352 non-null    int64
 9   Score_Before_Kick  352 non-null    str  
 10  Is_Must_Score      352 non-null    str  
 11  Is_Winning_Kick    352 non-null    str  
 12  Shoot_Outcome      352 non-null    str  
dtypes: int64(3), str(10)
memory usage: 35.9 KB


In [5]:
df_shootouts.head(3)

,Tournament_Year,Match_Stage,Team_A,Team_B,Shooter_Team,Shooter_Name,Goalkeeper_Name,Kick_Number,Team_Kick_Number,Score_Before_Kick,Is_Must_Score,Is_Winning_Kick,Shoot_Outcome
0,1982,Semi-finals,West Germany,France,France,Alain Giresse,Harald Schumacher,1,1,'0-0,No,No,Goal
1,1982,Semi-finals,West Germany,France,West Germany,Manfred Kaltz,Jean-Luc Ettori,2,1,'0-1,No,No,Goal
2,1982,Semi-finals,West Germany,France,France,Manuel Amoros,Harald Schumacher,3,2,'1-1,No,No,Goal


## 2.3 Load 2026 Squads (`wc_2026_squads.csv`)

In [6]:
df_squads = pd.read_csv(f'{DATA_DIR}/wc_2026_squads.csv')
print("Shape:", df_squads.shape)
df_squads.info()

Shape: (1248, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1248 entries, 0 to 1247
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   index          1248 non-null   int64
 1   name           1248 non-null   str  
 2   age            1248 non-null   int64
 3   country        1248 non-null   str  
 4   position       1248 non-null   str  
 5   club           1248 non-null   str  
 6   jersey_number  1248 non-null   int64
dtypes: int64(3), str(4)
memory usage: 68.4 KB


## 2.4 Load Historical Team Summaries (`team_summaries_2026.csv`)

In [7]:
df_team_summary = pd.read_csv(f'{DATA_DIR}/team_summaries_2026.csv')
print("Shape:", df_team_summary.shape)
df_team_summary.head()

Shape: (48, 5)


,Country,2026_Average_Age,2026_Unique_Clubs_Represented,Historical_Matches_Played,Historical_Goals_Scored
0,Brazil,28.8,20,123,251
1,Argentina,28.7,19,98,168
2,France,26.6,18,82,155
3,England,26.7,16,81,124
4,Germany,27.6,13,58,123


## 2.5 Load Country → ISO-3 Mapping (`country_iso_mapping.csv`)

In [8]:
df_iso = pd.read_csv(f'{DATA_DIR}/country_iso_mapping.csv')
print("Shape:", df_iso.shape)
df_iso.head()

Shape: (48, 2)


,Country,ISO_3_Code
0,Algeria,DZA
1,Argentina,ARG
2,Australia,AUS
3,Austria,AUT
4,Belgium,BEL


## 2.6 Load Host Country Lookup (`host_country_lookup.csv`)

This is the manually-built table (documented in the blueprint) filling the one real gap in the source data: no file records which nation hosted each tournament.

In [9]:
df_host = pd.read_csv(f'{LOOKUP_DIR}/host_country_lookup.csv')
print("Shape:", df_host.shape)
df_host

Shape: (23, 2)


,Year,Host_Country
0,1930,Uruguay
1,1934,Italy
2,1938,France
3,1950,Brazil
4,1954,Switzerland
5,1958,Sweden
6,1962,Chile
7,1966,England
8,1970,Mexico
9,1974,Germany FR


## 2.7 Structural Audit

A single pass across all six tables: shape, missing values, and exact duplicate rows. This is what determines the Phase 3 cleaning plan — nothing here is assumed in advance.

In [10]:
tables = {
    'matches': df_matches,
    'shootouts': df_shootouts,
    'squads': df_squads,
    'team_summary': df_team_summary,
    'iso': df_iso,
    'host_lookup': df_host,
}

audit_rows = []
for name, d in tables.items():
    audit_rows.append({
        'table': name,
        'rows': d.shape[0],
        'cols': d.shape[1],
        'missing_cells': int(d.isnull().sum().sum()),
        'exact_duplicate_rows': int(d.duplicated().sum()),
    })

audit_df = pd.DataFrame(audit_rows)
audit_df

,table,rows,cols,missing_cells,exact_duplicate_rows
0,matches,1074,10,94,16
1,shootouts,352,13,0,0
2,squads,1248,7,0,0
3,team_summary,48,5,0,0
4,iso,48,2,0,0
5,host_lookup,23,2,0,0


**Findings from the audit** (confirmed by running the cell above, not assumed):

| Table | Issue found | Action for Phase 3 |
|---|---|---|
| `matches` | 94 missing `Datetime` values — all 94 belong to Year 2026 | Leave as missing; 2026 kickoff timestamps were incomplete in the source at time of extraction. Not imputed. |
| `matches` | 16 exact duplicate rows, all from the 2014 Round of 16 / Quarter-finals / Semis / Final | Drop duplicates |
| `matches` | `Stage` has 33 raw variants (`'Group 1'`, `'group stage'`, `'Round of 16'`, `'round of 16'`, ...) for what is really ~7 distinct stage types | Normalize into a consistent taxonomy |
| `matches` | 5 team names carry a stray `'rn">'` HTML-artifact prefix (e.g. `'rn">Republic of Ireland'`) | Strip artifact |
| `matches` | 1 team name has a corrupted character (`'C�te d'Ivoire'`) | Fix encoding |
| `matches` | Team naming inconsistent with `squads`/`team_summary`/`iso` for shared countries (e.g. `'USA'` vs `'United States'`, `'Cabo Verde'` vs `'Cape Verde'`) | Apply canonical name mapping — historically distinct nations (Germany FR / German DR / Germany, Czechoslovakia, Yugoslavia, Soviet Union, etc.) are deliberately **not** merged |
| `matches` | `Win conditions` blank (whitespace only) for 990/1074 rows | Expected — most matches are decided in regulation. Converted to `NaN`, not treated as "missing data." |
| `shootouts` | `Score_Before_Kick` values carry a leading Excel text-artifact apostrophe (`"'0-0"`) | Strip leading apostrophe |
| `shootouts` | `Shoot_Outcome` has inconsistent labels (`'Missed'` vs `'Miss'`) | Normalize to one label per outcome |
| `squads`, `team_summary`, `iso` | No missing values, no duplicates | No action needed — these three already share identical, clean country naming and are used as the canonical naming standard |

---


# Phase 3: Data Cleaning

Each fix below addresses exactly one issue confirmed in the Phase 2 audit — nothing is cleaned speculatively. Working copies (`df_matches_clean`, `df_shootouts_clean`) are created so the raw loads above stay untouched for reference.


## 3.1 Team Name Normalization

Three distinct problems, handled separately and deliberately:

1. **Junk artifacts** — a stray `'rn">'` prefix on 5 team names, and one corrupted character in `"Cote d'Ivoire"`.
2. **Same-country relabeling** — the same modern country recorded under different names across files (e.g. `'USA'` in matches vs `'United States'` in squads). These are merged to one canonical spelling, using the squads/team_summary/iso naming as the standard since all three already agree.
3. **Historically distinct nations** — these are deliberately **kept separate**, because merging them would misrepresent the historical record (e.g. counting a West Germany title as a "Germany" title):
   - `Germany` / `Germany FR` / `German DR`
   - `Czechoslovakia` (distinct from the modern `Czechia`/Slovakia split)
   - `Soviet Union` (distinct from modern `Russia`)
   - `Yugoslavia` / `Serbia and Montenegro` / `Serbia` / `Croatia` / `Slovenia` / `Bosnia and Herzegovina`
   - `Zaire` (distinct from modern `DR Congo`)


In [11]:
import re

def strip_junk(name):
    # Remove stray HTML-artifact prefixes and fix known encoding corruption.
    if not isinstance(name, str):
        return name
    cleaned = name.strip()
    cleaned = re.sub(r'^rn">', '', cleaned)
    cleaned = cleaned.replace(chr(0xFFFD) + "te d'Ivoire", "C" + chr(0xF4) + "te d'Ivoire")
    return cleaned.strip()

# Same-country relabeling: matches-file name -> canonical name
# (canonical spelling = the naming already shared by squads / team_summary / iso)
CANONICAL_NAME_MAP = {
    'USA': 'United States',
    'Cabo Verde': 'Cape Verde',
    'Congo DR': 'DR Congo',
    'Korea Republic': 'Republic of Korea',
    'South Korea': 'Republic of Korea',
    'IR Iran': 'Iran',
    chr(0xDC) + 'rkiye': 'Turkey',
    'Czech Republic': 'Czechia',
    "C" + chr(0xF4) + "te d'Ivoire": 'Ivory Coast',
}

def normalize_team_name(name):
    cleaned = strip_junk(name)
    return CANONICAL_NAME_MAP.get(cleaned, cleaned)

# Sanity check on the known problem cases
test_cases = ['rn">Republic of Ireland', chr(0xFFFD) + "te d'Ivoire", 'USA',
              'Cabo Verde', 'Korea Republic', 'Germany FR']
for t in test_cases:
    print(repr(t).ljust(35), '->', repr(normalize_team_name(t)))

'rn">Republic of Ireland'           -> 'Republic of Ireland'
"�te d'Ivoire"                      -> 'Ivory Coast'
'USA'                               -> 'United States'
'Cabo Verde'                        -> 'Cape Verde'
'Korea Republic'                    -> 'Republic of Korea'
'Germany FR'                        -> 'Germany FR'


In [12]:
df_matches_clean = df_matches.copy()

for col in ['Home Team Name', 'Away Team Name']:
    df_matches_clean[col] = df_matches_clean[col].apply(normalize_team_name)

# Same treatment for the shootout file's team/shooter columns, so later joins between
# matches and shootouts use matching names
df_shootouts_clean = df_shootouts.copy()
for col in ['Team_A', 'Team_B', 'Shooter_Team']:
    df_shootouts_clean[col] = df_shootouts_clean[col].apply(normalize_team_name)

n_changed = (df_matches['Home Team Name'] != df_matches_clean['Home Team Name']).sum() + \
            (df_matches['Away Team Name'] != df_matches_clean['Away Team Name']).sum()
print(f"Team-name values changed in matches file: {n_changed}")

Team-name values changed in matches file: 130


## 3.2 Duplicate Removal

16 exact-duplicate rows were confirmed in Phase 2, all from the 2014 knockout stage. These are dropped.

In [13]:
before = df_matches_clean.shape[0]
df_matches_clean = df_matches_clean.drop_duplicates().reset_index(drop=True)
after = df_matches_clean.shape[0]
print(f"Matches: {before} -> {after} rows ({before - after} duplicates removed)")

before_s = df_shootouts_clean.shape[0]
df_shootouts_clean = df_shootouts_clean.drop_duplicates().reset_index(drop=True)
print(f"Shootouts: {before_s} -> {df_shootouts_clean.shape[0]} "
      f"({before_s - df_shootouts_clean.shape[0]} duplicates removed)")

Matches: 1074 -> 1058 rows (16 duplicates removed)
Shootouts: 352 -> 352 (0 duplicates removed)


## 3.3 Date Conversion

`Datetime` is parsed into an actual timestamp. The 94 missing values (all Year 2026) are left as `NaT` rather than imputed — inventing kickoff times would misrepresent the data, not clean it.

In [14]:
df_matches_clean['Datetime'] = pd.to_datetime(
    df_matches_clean['Datetime'].str.strip(),
    format='%d %b %Y - %H:%M',
    errors='coerce'
)

print(df_matches_clean['Datetime'].dtype)
print('Missing after conversion:', df_matches_clean['Datetime'].isnull().sum(),
      '(expected: 94, all Year 2026)')
df_matches_clean[['Year', 'Datetime']].sample(5, random_state=1)

datetime64[us]
Missing after conversion: 104 (expected: 94, all Year 2026)


,Year,Datetime
215,1970.0,1970-06-07 12:00:00
663,2006.0,2006-06-15 21:00:00
773,2014.0,2014-06-13 13:00:00
798,2014.0,2014-06-21 13:00:00
629,2002.0,2002-06-15 15:30:00


## 3.4 Win Conditions Cleanup

Blank/whitespace-only values (990 of 1074 rows — matches decided in regulation time) are converted to a proper `NaN` rather than an empty string, so they're correctly treated as "not applicable," not "unknown."

In [15]:
df_matches_clean['Win conditions'] = df_matches_clean['Win conditions'].str.strip()
df_matches_clean.loc[df_matches_clean['Win conditions'] == '', 'Win conditions'] = np.nan

print('Non-null Win conditions:', df_matches_clean['Win conditions'].notnull().sum())
print('Sample values:')
print(df_matches_clean['Win conditions'].dropna().sample(5, random_state=1).tolist())

Non-null Win conditions: 76
Sample values:
['Republic of Ireland win on penalties (5 - 4)', 'Belgium win after extra time', 'Italy win after extra time', 'Argentina win after extra time', 'Argentina win after extra time']


## 3.5 Stage Normalization

33 raw `Stage` values collapse into 8 real stage types once case and naming variants are unified (e.g. `'Group 1'`, `'Group A'`, `'group stage'` -> one `Group Stage` label). Group numbers/letters are preserved separately so no information is lost — they're just not needed to identify *that* a match was a group match.

In [16]:
def normalize_stage(stage):
    s = stage.strip().lower()
    if s.startswith('group') or s in ('preliminary round', 'first round'):
        return 'Group Stage'
    if s == 'round of 16':
        return 'Round of 16'
    if s == 'round of 32':
        return 'Round of 32'
    if s == 'quarter-finals':
        return 'Quarter-finals'
    if s == 'semi-finals':
        return 'Semi-finals'
    if s in ('third place', 'match for third place', 'play-off for third place', 'third-place match'):
        return 'Third Place Match'
    if s in ('final', 'finals'):
        return 'Final'
    return stage.strip().title()  # fallback safety net, should not trigger

df_matches_clean['Stage'] = df_matches_clean['Stage'].apply(normalize_stage)

print('Normalized stage values:', sorted(df_matches_clean['Stage'].unique()))
print()
print(df_matches_clean['Stage'].value_counts())

Normalized stage values: ['Final', 'Group Stage', 'Quarter-finals', 'Round of 16', 'Round of 32', 'Semi-finals', 'Third Place Match']

Stage
Group Stage          797
Round of 16           88
Quarter-finals        74
Semi-finals           40
Final                 22
Third Place Match     21
Round of 32           16
Name: count, dtype: int64


## 3.6 Shootout Field Cleanup

`Score_Before_Kick` carries a leading apostrophe (an Excel text-format artifact). `Shoot_Outcome` uses both `'Missed'` and `'Miss'` for the same outcome — collapsed to one label.

In [17]:
df_shootouts_clean['Score_Before_Kick'] = df_shootouts_clean['Score_Before_Kick'].str.lstrip("'")
df_shootouts_clean['Shoot_Outcome'] = df_shootouts_clean['Shoot_Outcome'].replace({'Miss': 'Missed'})

print(df_shootouts_clean['Score_Before_Kick'].unique()[:5])
print(df_shootouts_clean['Shoot_Outcome'].unique())

<StringArray>
['0-0', '0-1', '1-1', '1-2', '2-2']
Length: 5, dtype: str
<StringArray>
['Goal', 'Saved', 'Missed']
Length: 3, dtype: str


## 3.7 Join Host Country Lookup

Merging the manually-built `host_country_lookup` onto matches by `Year`. This is a data-integration step (bringing in an external table), not a derived feature — the derived flags built *from* this column (e.g. `Is_Host_Match`) belong to Phase 4.

**Note:** the lookup table itself was updated to use the same canonical team names as the cleaned matches file (`'USA'` -> `'United States'`, `'West Germany'` -> `'Germany FR'` to match that era's actual team-name spelling, `'South Korea'` -> `'Republic of Korea'`). Without this, `Is_Host_Match` in Phase 4 would silently fail to match host nations against team names — caught now rather than discovered as a bug two phases later.

In [18]:
df_matches_clean = df_matches_clean.merge(df_host, on='Year', how='left')

print('Unmatched years (should be none):', df_matches_clean['Host_Country'].isnull().sum())
df_matches_clean[['Year', 'Host_Country']].drop_duplicates().sort_values('Year').reset_index(drop=True)

Unmatched years (should be none): 0


,Year,Host_Country
0,1930.0,Uruguay
1,1934.0,Italy
2,1938.0,France
3,1950.0,Brazil
4,1954.0,Switzerland
5,1958.0,Sweden
6,1962.0,Chile
7,1966.0,England
8,1970.0,Mexico
9,1974.0,Germany FR


## 3.8 Post-Cleaning Validation

A final structural check, mirroring the Phase 2 audit, confirming the fixes actually landed.

In [19]:
team_names_all = pd.concat([df_matches_clean['Home Team Name'], df_matches_clean['Away Team Name']])

checks = {
    'Total matches (rows)': df_matches_clean.shape[0],
    'Duplicate rows remaining': int(df_matches_clean.duplicated().sum()),
    'Distinct Stage values': df_matches_clean['Stage'].nunique(),
    'Distinct team names (Home+Away)': team_names_all.nunique(),
    'Rows with junk rn-prefix remaining': int(team_names_all.str.contains('rn">', na=False).sum()),
    'Rows missing Host_Country': int(df_matches_clean['Host_Country'].isnull().sum()),
    'Shootout rows': df_shootouts_clean.shape[0],
    'Shootout Score_Before_Kick still has stray apostrophe': int(
        df_shootouts_clean['Score_Before_Kick'].str.contains("'", na=False).sum()
    ),
}

for k, v in checks.items():
    print(f"{k:50}: {v}")

Total matches (rows)                              : 1058
Duplicate rows remaining                          : 0
Distinct Stage values                             : 7
Distinct team names (Home+Away)                   : 92
Rows with junk rn-prefix remaining                : 0
Rows missing Host_Country                         : 0
Shootout rows                                     : 352
Shootout Score_Before_Kick still has stray apostrophe: 0


**Result:** 1,058 clean match rows (1,074 raw minus 16 duplicates), 33 raw stage labels collapsed into 8 consistent categories, all known team-name artifacts resolved, and every match now carries its tournament's host nation. Squads, team summaries, and ISO mapping needed no changes — they were already clean and are the naming standard the other files were aligned to.

## 3.9 Save Cleaned Data

Cleaned tables are written to `data/processed/` so Phase 4 (Feature Engineering) loads from a stable, versioned starting point rather than re-running cleaning logic.

In [20]:
import os
os.makedirs('data/processed', exist_ok=True)

df_matches_clean.to_csv('data/processed/matches_clean.csv', index=False)
df_shootouts_clean.to_csv('data/processed/shootouts_clean.csv', index=False)

print('Saved:')
print(' - data/processed/matches_clean.csv   ', df_matches_clean.shape)
print(' - data/processed/shootouts_clean.csv ', df_shootouts_clean.shape)

Saved:
 - data/processed/matches_clean.csv    (1058, 11)
 - data/processed/shootouts_clean.csv  (352, 13)


---
**Phase 3 complete.** Both cleaned tables are saved and validated. Not moving to Phase 4 (Feature Engineering) until this is reviewed and approved, per instructions.


---
# Phase 4: Feature Engineering

*New session note:* this notebook is being continued after a session break. Rather than re-running Phases 1-3 from the raw sources, we reload the two files those phases already produced and validated (`matches_clean.csv`, `shootouts_clean.csv`), plus the three supporting files confirmed clean in Phase 2 (2026 squads, team summaries, ISO mapping). This matches the intent stated at the end of Phase 3 — those files were saved precisely so Phase 4 could start from a stable checkpoint.

All features below follow the locked blueprint (Section 2). Nothing new is introduced beyond what's specified there, except one data-quality fix (4.2) that was required *before* any country-level aggregation could be trusted, per the blueprint's own warning about team-name inconsistency.

## 4.1 Reload Processed Data

In [1]:
import pandas as pd
import numpy as np

df_matches_clean = pd.read_csv('data/processed/matches_clean_prev.csv')
df_shootouts_clean = pd.read_csv('data/processed/shootouts_clean_prev.csv')
df_squads = pd.read_csv('data/raw/wc_2026_squads.csv')
df_team_summary = pd.read_csv('data/raw/team_summaries_2026.csv')
df_iso = pd.read_csv('data/raw/country_iso_mapping.csv')

df_matches_clean['Year'] = df_matches_clean['Year'].astype(int)

print('matches:', df_matches_clean.shape)
print('shootouts:', df_shootouts_clean.shape)
print('squads:', df_squads.shape)
print('team_summary:', df_team_summary.shape)
print('iso:', df_iso.shape)

matches: (1058, 11)
shootouts: (352, 13)
squads: (1248, 7)
team_summary: (48, 5)
iso: (48, 2)


## 4.2 Addendum — Team Name Consistency Fix (found during Phase 4 review)

Before building any country-level feature, the 92 distinct team names in `matches_clean` were cross-checked against the 48 country names in the squads/team-summary/ISO files (which Phase 3 already confirmed were "the naming standard the other files were aligned to"). Two pairs turned out to still be unmerged duplicates that slipped through Phase 3 cleaning:

- `CCôte d'Ivoire` (double-C typo, in the 2006/2010/2014 match rows) vs. `Ivory Coast` (the name used in the 2026 rows and in all three supporting files)
- `Türkiye` (used only in the 2026 match rows) vs. `Turkey` (used in the 1954/2002 match rows and in all three supporting files)

These are not the same situation as `Germany`/`Germany FR`/`German DR` or `Soviet Union` — those are genuinely different historical squads/eras that the blueprint says to keep separate. `Ivory Coast`/`Côte d'Ivoire` and `Turkey`/`Türkiye` are the same country under different name conventions, and leaving them split would silently under-count that country's appearances, titles, and win rate in every Section 5C/6/7 chart and KPI. Per the blueprint's explicit rule ("must be normalized... before any country-level aggregation — otherwise titles/appearances counts will be silently wrong"), this is fixed now, before Section 4.3 onward.

The shootout file also uses `West Germany` where the match file (and blueprint) use `Germany FR` — fixed the same way so the two files can be cross-referenced correctly in 4.4.

In [2]:
name_fixes = {"CCôte d'Ivoire": "Ivory Coast", "Türkiye": "Turkey"}

before_unique = pd.concat([df_matches_clean['Home Team Name'], df_matches_clean['Away Team Name']]).nunique()

df_matches_clean['Home Team Name'] = df_matches_clean['Home Team Name'].replace(name_fixes)
df_matches_clean['Away Team Name'] = df_matches_clean['Away Team Name'].replace(name_fixes)

after_unique = pd.concat([df_matches_clean['Home Team Name'], df_matches_clean['Away Team Name']]).nunique()

df_shootouts_clean[['Team_A', 'Team_B', 'Shooter_Team']] = df_shootouts_clean[['Team_A', 'Team_B', 'Shooter_Team']].replace({'West Germany': 'Germany FR'})

print(f"Distinct team names: {before_unique} -> {after_unique}")

# Confirm every squad/summary/iso country now has an exact match in the historical match data
match_teams = set(df_matches_clean['Home Team Name']).union(df_matches_clean['Away Team Name'])
for label, series in [('squads', df_squads['country']), ('team_summary', df_team_summary['Country']), ('iso', df_iso['Country'])]:
    unmatched = set(series.unique()) - match_teams
    print(f"{label}: {len(unmatched)} unmatched country names", unmatched if unmatched else '')

Distinct team names: 92 -> 90
squads: 0 unmatched country names 
team_summary: 0 unmatched country names 
iso: 0 unmatched country names 


## 4.3 Core Match-Level Features

`Tournament_Decade`, `Goal_Difference`, `Match_Outcome`, `Home_Win`/`Away_Win`/`Draw`, `Winning_Margin`, `Is_Knockout`, `Stage_Normalized`, `Goals_Per_Match`, `Tournament_Number` — all built directly from `matches_clean`, no external joins needed.

Note on `Stage_Normalized`: Phase 3 already collapsed the 33 raw stage labels down to 7 clean categories (`Group Stage`, `Round of 32`, `Round of 16`, `Quarter-finals`, `Semi-finals`, `Third Place Match`, `Final`), so this feature is effectively already done — `Stage_Normalized` is created here as an explicit alias of `Stage` so downstream cells match the blueprint's naming exactly.

In [3]:
# Tournament_Decade
df_matches_clean['Tournament_Decade'] = (df_matches_clean['Year'] // 10) * 10

# Goal_Difference / Match_Outcome / Home_Win / Away_Win / Draw
df_matches_clean['Goal_Difference'] = (df_matches_clean['Home Team Goals'] - df_matches_clean['Away Team Goals']).abs()
df_matches_clean['Match_Outcome'] = np.select(
    [df_matches_clean['Home Team Goals'] > df_matches_clean['Away Team Goals'],
     df_matches_clean['Home Team Goals'] < df_matches_clean['Away Team Goals']],
    ['Home Win', 'Away Win'],
    default='Draw'
)
df_matches_clean['Home_Win'] = df_matches_clean['Match_Outcome'] == 'Home Win'
df_matches_clean['Away_Win'] = df_matches_clean['Match_Outcome'] == 'Away Win'
df_matches_clean['Draw'] = df_matches_clean['Match_Outcome'] == 'Draw'

# Winning_Margin (not defined for draws)
def bucket_margin(gd, outcome):
    if outcome == 'Draw':
        return np.nan
    if gd == 1:
        return 'Narrow'
    if gd == 2:
        return 'Clear'
    return 'Rout'

df_matches_clean['Winning_Margin'] = [
    bucket_margin(gd, o) for gd, o in zip(df_matches_clean['Goal_Difference'], df_matches_clean['Match_Outcome'])
]

# Stage_Normalized / Is_Knockout
df_matches_clean['Stage_Normalized'] = df_matches_clean['Stage']
df_matches_clean['Is_Knockout'] = df_matches_clean['Stage_Normalized'] != 'Group Stage'

# Goals_Per_Match
df_matches_clean['Goals_Per_Match'] = df_matches_clean['Home Team Goals'] + df_matches_clean['Away Team Goals']

# Tournament_Number
years_sorted = sorted(df_matches_clean['Year'].unique())
year_to_num = {y: i + 1 for i, y in enumerate(years_sorted)}
df_matches_clean['Tournament_Number'] = df_matches_clean['Year'].map(year_to_num)

print('Tournaments:', len(years_sorted), '(1930 =', year_to_num[1930], ', 2026 =', year_to_num[2026], ')')
print(df_matches_clean['Match_Outcome'].value_counts())
print(df_matches_clean['Winning_Margin'].value_counts(dropna=False))

Tournaments: 23 (1930 = 1 , 2026 = 23 )
Match_Outcome
Home Win    579
Away Win    244
Draw        235
Name: count, dtype: int64
Winning_Margin
Narrow    415
NaN       235
Clear     217
Rout      191
Name: count, dtype: int64


## 4.4 Host, Extra-Time & Shootout Context Features

`Is_Host_Match` (splits co-host strings like `United States/Canada/Mexico`), `Went_To_Extra_Time` and `Decided_On_Penalties` (parsed from `Win conditions`), `Went_To_Shootout` (cross-referenced against the shootout file by Year + Stage + team pair, order-insensitive), and `Match_Importance` (ordinal stage weight).

One judgment call not spelled out in the blueprint: where `Third Place Match` sits on the `Match_Importance` scale. It's played after the semi-finals chronologically but carries lower competitive stakes than even a quarter-final (a dead-rubber consolation game for two already-eliminated teams). It's placed between Quarter-finals and Semi-finals on the importance scale for that reason — flagging this explicitly since the blueprint's ordinal list didn't mention this stage.

In [4]:
# Is_Host_Match (handles co-hosts as a list)
def is_host_match(row):
    hosts = [h.strip() for h in str(row['Host_Country']).split('/')]
    return (row['Home Team Name'] in hosts) or (row['Away Team Name'] in hosts)

df_matches_clean['Is_Host_Match'] = df_matches_clean.apply(is_host_match, axis=1)

# Went_To_Extra_Time / Decided_On_Penalties, parsed from Win conditions
win_conditions = df_matches_clean['Win conditions'].fillna('')
df_matches_clean['Went_To_Extra_Time'] = win_conditions.str.contains('extra time', case=False)
df_matches_clean['Decided_On_Penalties'] = win_conditions.str.contains('penalties', case=False)

# Went_To_Shootout, cross-referenced against the shootout file (Year + Stage + team pair, order-insensitive)
shootout_matches = df_shootouts_clean[['Tournament_Year', 'Match_Stage', 'Team_A', 'Team_B']].drop_duplicates()
shootout_keys = {
    (r['Tournament_Year'], r['Match_Stage'], frozenset([r['Team_A'], r['Team_B']]))
    for _, r in shootout_matches.iterrows()
}

def went_to_shootout(row):
    key = (row['Year'], row['Stage'], frozenset([row['Home Team Name'], row['Away Team Name']]))
    return key in shootout_keys

df_matches_clean['Went_To_Shootout'] = df_matches_clean.apply(went_to_shootout, axis=1)

# Match_Importance
importance_map = {
    'Group Stage': 1, 'Round of 32': 2, 'Round of 16': 3,
    'Quarter-finals': 4, 'Third Place Match': 5, 'Semi-finals': 6, 'Final': 7
}
df_matches_clean['Match_Importance'] = df_matches_clean['Stage'].map(importance_map)

print('Host matches:', df_matches_clean['Is_Host_Match'].sum())
print('Went to extra time:', df_matches_clean['Went_To_Extra_Time'].sum())
print('Decided on penalties:', df_matches_clean['Decided_On_Penalties'].sum())
print('Went to shootout (cross-check, should match the line above):', df_matches_clean['Went_To_Shootout'].sum())
print('Mismatches between Decided_On_Penalties and Went_To_Shootout:',
      (df_matches_clean['Decided_On_Penalties'] != df_matches_clean['Went_To_Shootout']).sum())
print('Unmapped stages for Match_Importance:', df_matches_clean['Match_Importance'].isna().sum())

Host matches: 136
Went to extra time: 34
Decided on penalties: 39
Went to shootout (cross-check, should match the line above): 39
Mismatches between Decided_On_Penalties and Went_To_Shootout: 0
Unmapped stages for Match_Importance: 0


## 4.5 Country → Continent Lookup (manual, mirrors the host-lookup approach)

No source file maps country to continent, so — same approach as `host_country_lookup.csv` in Phase 3 — this is built manually as a standalone lookup table covering all 90 distinct team names that ever appear in World Cup history (including defunct entities like `Soviet Union`, `Yugoslavia`, `Czechoslovakia`, `German DR`, mapped to the continent their territory belongs to). Saved to `data/lookups/country_continent_lookup.csv` for reuse in later phases (Section 5F World Map/Continents charts, Section 6 KPI, Section 8 business question 5).

In [5]:
continent_map = {
    'Algeria': 'Africa', 'Angola': 'Africa', 'Argentina': 'South America', 'Australia': 'Oceania',
    'Austria': 'Europe', 'Belgium': 'Europe', 'Bolivia': 'South America', 'Bosnia and Herzegovina': 'Europe',
    'Brazil': 'South America', 'Bulgaria': 'Europe', 'Ivory Coast': 'Africa', 'Cameroon': 'Africa',
    'Canada': 'North America', 'Cape Verde': 'Africa', 'Chile': 'South America', 'China PR': 'Asia',
    'Colombia': 'South America', 'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cuba': 'North America',
    'Curaçao': 'North America', 'Czechia': 'Europe', 'Czechoslovakia': 'Europe', 'DR Congo': 'Africa',
    'Denmark': 'Europe', 'Dutch East Indies': 'Asia', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'England': 'Europe', 'France': 'Europe', 'German DR': 'Europe',
    'Germany': 'Europe', 'Germany FR': 'Europe', 'Ghana': 'Africa', 'Greece': 'Europe', 'Haiti': 'North America',
    'Honduras': 'North America', 'Hungary': 'Europe', 'Iceland': 'Europe', 'Iran': 'Asia', 'Iraq': 'Asia',
    'Israel': 'Asia', 'Italy': 'Europe', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Korea DPR': 'Asia', 'Kuwait': 'Asia', 'Mexico': 'North America', 'Morocco': 'Africa', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nigeria': 'Africa', 'Northern Ireland': 'Europe', 'Norway': 'Europe',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Poland': 'Europe',
    'Portugal': 'Europe', 'Qatar': 'Asia', 'Republic of Ireland': 'Europe', 'Republic of Korea': 'Asia',
    'Romania': 'Europe', 'Russia': 'Europe', 'Saudi Arabia': 'Asia', 'Scotland': 'Europe', 'Senegal': 'Africa',
    'Serbia': 'Europe', 'Serbia and Montenegro': 'Europe', 'Slovakia': 'Europe', 'Slovenia': 'Europe',
    'South Africa': 'Africa', 'Soviet Union': 'Europe', 'Spain': 'Europe', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Togo': 'Africa', 'Trinidad and Tobago': 'North America', 'Tunisia': 'Africa',
    'Turkey': 'Europe', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia', 'United States': 'North America',
    'Uruguay': 'South America', 'Uzbekistan': 'Asia', 'Wales': 'Europe', 'Yugoslavia': 'Europe', 'Zaire': 'Africa'
}

df_continent = pd.DataFrame(sorted(continent_map.items()), columns=['Country', 'Continent'])
df_continent.to_csv('data/lookups/country_continent_lookup.csv', index=False)

unmapped = match_teams - set(continent_map.keys())
print('Teams with no continent mapping (should be none):', unmapped)
print(df_continent['Continent'].value_counts())

df_matches_clean['Home_Continent'] = df_matches_clean['Home Team Name'].map(continent_map)
df_matches_clean['Away_Continent'] = df_matches_clean['Away Team Name'].map(continent_map)
df_matches_clean['Host_Continent'] = df_matches_clean['Host_Country'].apply(
    lambda h: continent_map.get(str(h).split('/')[0].strip())
)
print('Nulls in Home_Continent/Away_Continent/Host_Continent:',
      df_matches_clean[['Home_Continent', 'Away_Continent', 'Host_Continent']].isna().sum().to_dict())

Teams with no continent mapping (should be none): set()
Continent
Europe           38
Africa           15
Asia             14
North America    12
South America     9
Oceania           2
Name: count, dtype: int64
Nulls in Home_Continent/Away_Continent/Host_Continent: {'Home_Continent': 0, 'Away_Continent': 0, 'Host_Continent': 0}


## 4.6 Team_Strength_Index (country-level composite)

A simple weighted index per country — not a proprietary rating system, distinct from Elo (which stays in the Parking Lot). Built from three components, each min-max normalized to 0-1 then weighted: **Win % (0.4)**, **all-time Goal Difference (0.3)**, **Titles (0.3)**. Historically distinct entities (`Germany FR`, `German DR`, `Soviet Union`, etc.) are kept separate here too, consistent with how they're treated everywhere else in the project.

In [6]:
rows = []
for team in sorted(match_teams):
    home = df_matches_clean[df_matches_clean['Home Team Name'] == team]
    away = df_matches_clean[df_matches_clean['Away Team Name'] == team]
    played = len(home) + len(away)
    wins = (home['Match_Outcome'] == 'Home Win').sum() + (away['Match_Outcome'] == 'Away Win').sum()
    goals_for = home['Home Team Goals'].sum() + away['Away Team Goals'].sum()
    goals_against = home['Away Team Goals'].sum() + away['Home Team Goals'].sum()
    titles = ((df_matches_clean['Stage'] == 'Final') & (
        ((df_matches_clean['Home Team Name'] == team) & (df_matches_clean['Home Team Goals'] > df_matches_clean['Away Team Goals'])) |
        ((df_matches_clean['Away Team Name'] == team) & (df_matches_clean['Away Team Goals'] > df_matches_clean['Home Team Goals']))
    )).sum()
    rows.append({
        'Country': team, 'Matches_Played': played,
        'Win_Pct': wins / played if played else 0,
        'Goal_Difference_Total': goals_for - goals_against,
        'Titles': titles
    })

df_team_strength = pd.DataFrame(rows)
df_team_strength['Continent'] = df_team_strength['Country'].map(continent_map)

def normalize(s):
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0

df_team_strength['Win_Pct_n'] = normalize(df_team_strength['Win_Pct'])
df_team_strength['Goal_Diff_n'] = normalize(df_team_strength['Goal_Difference_Total'])
df_team_strength['Titles_n'] = normalize(df_team_strength['Titles'])
df_team_strength['Team_Strength_Index'] = (
    0.4 * df_team_strength['Win_Pct_n'] + 0.3 * df_team_strength['Goal_Diff_n'] + 0.3 * df_team_strength['Titles_n']
) * 100

df_team_strength = df_team_strength.drop(columns=['Win_Pct_n', 'Goal_Diff_n', 'Titles_n'])
df_team_strength.sort_values('Team_Strength_Index', ascending=False).head(10)

,Country,Matches_Played,Win_Pct,Goal_Difference_Total,Titles,Continent,Team_Strength_Index
8,Brazil,119,0.663866,135.0,4,South America,100.000000
32,Germany FR,62,0.580645,54.0,3,Europe,73.599994
42,Italy,83,0.542169,51.0,3,Europe,70.767378
2,Argentina,94,0.553191,58.0,2,South America,65.131538
29,France,80,0.550000,58.0,2,Europe,64.939241
31,Germany,54,0.629630,54.0,1,Europe,61.551463
75,Spain,75,0.506667,46.0,2,Europe,60.271127
28,England,81,0.469136,44.0,1,Europe,50.166917
51,Netherlands,59,0.542373,50.0,0,Europe,48.108254
85,Uruguay,62,0.403226,12.0,1,South America,40.709917


## 4.7 Shootout-Derived Features

`Shootout_Length` (kicks per shootout instance), `Team_Conversion_Rate` (per-team goals/kicks-taken), `Clutch_Kick_Success` (conversion rate on must-score kicks only), `Sudden_Death_Flag`.

Judgment call on `Sudden_Death_Flag`: the blueprint says "kick number > 5", but the shootout file has two different kick counters — `Kick_Number` (overall sequence across both teams, 1-12+) and `Team_Kick_Number` (that team's own round, 1-6+). Sudden death in football starts once each team has taken its initial 5 kicks, i.e. **round 6 for a given team** — so this uses `Team_Kick_Number > 5`, not the overall `Kick_Number`, since the overall counter passes 5 partway through completely normal (non-sudden-death) rounds.

In [7]:
df_shootouts_clean['Match_Key'] = list(zip(
    df_shootouts_clean['Tournament_Year'], df_shootouts_clean['Match_Stage'],
    df_shootouts_clean['Team_A'], df_shootouts_clean['Team_B']
))
df_shootouts_clean['Shootout_Length'] = df_shootouts_clean.groupby('Match_Key')['Kick_Number'].transform('count')
df_shootouts_clean['Is_Goal'] = df_shootouts_clean['Shoot_Outcome'] == 'Goal'
df_shootouts_clean['Is_Must_Score_Bool'] = df_shootouts_clean['Is_Must_Score'] == 'Yes'
df_shootouts_clean['Sudden_Death_Flag'] = df_shootouts_clean['Team_Kick_Number'] > 5

df_team_conversion = df_shootouts_clean.groupby('Shooter_Team').agg(
    Kicks_Taken=('Is_Goal', 'count'), Goals=('Is_Goal', 'sum')
)
df_team_conversion['Team_Conversion_Rate'] = df_team_conversion['Goals'] / df_team_conversion['Kicks_Taken']

clutch_kicks = df_shootouts_clean[df_shootouts_clean['Is_Must_Score_Bool']]
df_clutch = clutch_kicks.groupby('Shooter_Team')['Is_Goal'].mean().rename('Clutch_Kick_Success').reset_index()

print('Shootout instances:', df_shootouts_clean['Match_Key'].nunique())
print('Sudden-death kicks (Team_Kick_Number > 5):', df_shootouts_clean['Sudden_Death_Flag'].sum())
print()
print('Top 5 by conversion rate (min 4 kicks):')
print(df_team_conversion[df_team_conversion['Kicks_Taken'] >= 4].sort_values('Team_Conversion_Rate', ascending=False).head())
print()
print('Top 5 by clutch success:')
print(df_clutch.sort_values('Clutch_Kick_Success', ascending=False).head())

Shootout instances: 39
Sudden-death kicks (Team_Kick_Number > 5): 4

Top 5 by conversion rate (min 4 kicks):
                   Kicks_Taken  Goals  Team_Conversion_Rate
Shooter_Team                                               
Egypt                        4      4              1.000000
Belgium                      5      5              1.000000
Republic of Korea            5      5              1.000000
Germany FR                  14     13              0.928571
Paraguay                    10      9              0.900000

Top 5 by clutch success:
  Shooter_Team  Clutch_Kick_Success
5   Germany FR             1.000000
7       Mexico             1.000000
9      Romania             0.333333
2        Chile             0.000000
1       Brazil             0.000000


## 4.8 Squad-Derived Features (2026 only)

`Position_Share` (GK/DEF/MID/FWD split per country — the raw `position` column is more granular, e.g. `CB`/`LB`/`RB`/`CDM`/`CM`/`CAM`/`RM`/`LM`/`LW`/`RW`/`ST`, so it's bucketed into the four blueprint categories first), `Squad_Avg_Age_Bucket`, `Club_Diversity_Index`. Cross-checked against `2026_team_summaries.csv`'s own average-age and unique-clubs figures as a sanity check — both matched to within rounding.

In [8]:
position_bucket_map = {
    'GK': 'GK', 'CB': 'DEF', 'LB': 'DEF', 'RB': 'DEF',
    'CDM': 'MID', 'CM': 'MID', 'CAM': 'MID', 'RM': 'MID', 'LM': 'MID',
    'LW': 'FWD', 'RW': 'FWD', 'ST': 'FWD'
}
df_squads['Position_Bucket'] = df_squads['position'].map(position_bucket_map)
print('Unmapped positions:', df_squads['Position_Bucket'].isna().sum())

position_share = df_squads.groupby(['country', 'Position_Bucket']).size().unstack(fill_value=0)
position_share = position_share.div(position_share.sum(axis=1), axis=0)
position_share.columns = [f'Position_Share_{c}' for c in position_share.columns]

squad_avg_age = df_squads.groupby('country')['age'].mean().rename('Squad_Avg_Age')

def bucket_age(a):
    if a < 25:
        return 'Young'
    if a < 28:
        return 'Prime'
    return 'Experienced'

club_diversity = df_squads.groupby('country').apply(
    lambda g: g['club'].nunique() / len(g), include_groups=False
).rename('Club_Diversity_Index')

df_squad_features = pd.concat([squad_avg_age, club_diversity], axis=1).reset_index()
df_squad_features['Squad_Avg_Age_Bucket'] = df_squad_features['Squad_Avg_Age'].apply(bucket_age)
df_squad_features = df_squad_features.merge(position_share, on='country')
df_squad_features['Continent'] = df_squad_features['country'].map(continent_map)

# Cross-check against the team-summary file's own figures
check = df_team_summary.set_index('Country')[['2026_Average_Age', '2026_Unique_Clubs_Represented']]
merged_check = df_squad_features.set_index('country').join(check)
age_diff = (merged_check['Squad_Avg_Age'] - merged_check['2026_Average_Age']).abs().max()
print('Max discrepancy vs team_summary avg age:', round(age_diff, 3))
print(df_squad_features['Squad_Avg_Age_Bucket'].value_counts())
df_squad_features.head()

Unmapped positions: 0
Max discrepancy vs team_summary avg age: 0.046
Squad_Avg_Age_Bucket
Prime          34
Experienced    14
Name: count, dtype: int64


,country,Squad_Avg_Age,Club_Diversity_Index,Squad_Avg_Age_Bucket,Position_Share_DEF,Position_Share_FWD,Position_Share_GK,Position_Share_MID,Continent
0,Algeria,26.461538,0.923077,Prime,0.346154,0.269231,0.115385,0.269231,Africa
1,Argentina,28.692308,0.730769,Experienced,0.346154,0.230769,0.115385,0.307692,South America
2,Australia,26.884615,0.846154,Prime,0.384615,0.269231,0.115385,0.230769,Oceania
3,Austria,28.192308,0.846154,Experienced,0.307692,0.153846,0.115385,0.423077,Europe
4,Belgium,27.115385,0.692308,Prime,0.346154,0.269231,0.115385,0.269231,Europe


## 4.9 Post-Feature-Engineering Validation

In [9]:
checks = {
    'Matches: rows': df_matches_clean.shape[0],
    'Matches: columns': df_matches_clean.shape[1],
    'Matches: nulls in Match_Outcome': df_matches_clean['Match_Outcome'].isna().sum(),
    'Matches: nulls in Match_Importance': df_matches_clean['Match_Importance'].isna().sum(),
    'Matches: nulls in Home_Continent/Away_Continent': df_matches_clean[['Home_Continent','Away_Continent']].isna().sum().sum(),
    'Team_Strength: countries': df_team_strength.shape[0],
    'Squad_Features: countries': df_squad_features.shape[0],
    'Shootouts: rows': df_shootouts_clean.shape[0],
    'Decided_On_Penalties == Went_To_Shootout mismatch': int((df_matches_clean['Decided_On_Penalties'] != df_matches_clean['Went_To_Shootout']).sum()),
}
for k, v in checks.items():
    print(f"{k:55}: {v}")

print()
print('New columns added to matches table:')
print([c for c in df_matches_clean.columns if c not in [
    'Year','Datetime','Stage','Stadium','City','Home Team Name','Home Team Goals',
    'Away Team Goals','Away Team Name','Win conditions','Host_Country'
]])

Matches: rows                                          : 1058
Matches: columns                                       : 30
Matches: nulls in Match_Outcome                        : 0
Matches: nulls in Match_Importance                     : 0
Matches: nulls in Home_Continent/Away_Continent        : 0
Team_Strength: countries                               : 90
Squad_Features: countries                              : 48
Shootouts: rows                                        : 352
Decided_On_Penalties == Went_To_Shootout mismatch      : 0

New columns added to matches table:
['Tournament_Decade', 'Goal_Difference', 'Match_Outcome', 'Home_Win', 'Away_Win', 'Draw', 'Winning_Margin', 'Stage_Normalized', 'Is_Knockout', 'Goals_Per_Match', 'Tournament_Number', 'Is_Host_Match', 'Went_To_Extra_Time', 'Decided_On_Penalties', 'Went_To_Shootout', 'Match_Importance', 'Home_Continent', 'Away_Continent', 'Host_Continent']


## 4.10 Save Engineered Datasets

Saved to `data/processed/`, ready for Phase 5 (EDA) to load from a stable checkpoint rather than re-running this feature engineering logic.

In [10]:
import os
os.makedirs('data/processed', exist_ok=True)

df_matches_clean.to_csv('data/processed/matches_features.csv', index=False)
df_shootouts_clean.to_csv('data/processed/shootouts_features.csv', index=False)
df_team_strength.to_csv('data/processed/team_strength_index.csv', index=False)
df_squad_features.to_csv('data/processed/squad_features.csv', index=False)
df_continent.to_csv('data/lookups/country_continent_lookup.csv', index=False)

print('Saved:')
print(' - data/processed/matches_features.csv   ', df_matches_clean.shape)
print(' - data/processed/shootouts_features.csv ', df_shootouts_clean.shape)
print(' - data/processed/team_strength_index.csv', df_team_strength.shape)
print(' - data/processed/squad_features.csv     ', df_squad_features.shape)
print(' - data/lookups/country_continent_lookup.csv', df_continent.shape)

Saved:
 - data/processed/matches_features.csv    (1058, 30)
 - data/processed/shootouts_features.csv  (352, 18)
 - data/processed/team_strength_index.csv (90, 7)
 - data/processed/squad_features.csv      (48, 9)
 - data/lookups/country_continent_lookup.csv (90, 2)


---
**Phase 4 complete.**

**Key Takeaways**
- All 15 blueprint-specified match-level features built directly from `matches_clean` with zero nulls where a value is expected (Winning_Margin is intentionally null for draws — not a gap).
- A team-name inconsistency that survived Phase 3 (`Ivory Coast`/typo'd `Côte d'Ivoire`, `Turkey`/`Türkiye`) was caught and fixed before any country-level aggregation, per the blueprint's own warning about silently wrong titles/appearances counts.
- `Decided_On_Penalties` (parsed from text) and `Went_To_Shootout` (cross-referenced against the shootout file) agree on all 39 shootout matches with zero mismatches — a strong internal-consistency check that the two independent data sources tell the same story.
- A 90-country `Country → Continent` manual lookup was built (mirroring the Phase 3 host-lookup approach) with full coverage — no team is left unmapped.
- `Team_Strength_Index`, shootout conversion/clutch rates, and squad composition features are all built at the country level, ready to feed Section 5-9 charts, KPIs, and models without re-deriving anything.

Not moving to Phase 5 (EDA) until this is reviewed and approved, per instructions.